# Example: Single spin Part 2: Gradient descent gate optimization

First, we make the necessary imports.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np

from cthree.OptimisationMap import OptimisationMap
from cthree.Quantity import Quantity
from cthree.measurement.UnitaryFidelity import UnitaryFidelity
from cthree.model.GeneratorDrive import GeneratorDrive
from cthree.model.Qubit import Qubit
from cthree.propagation.ScipyExpmGOAT import ScipyExpmGOAT
from cthree.optimisers.ScipyOptimiserGradient import ScipyOptimiserGradient

from cthree.model.ClosedModel import ClosedModel
from cthree.model.Hamiltonian import Hamiltonian

from cthree.signal.SimpleGenerator import CosGenerator
from cthree.signal.Device import CosTone

For signal generation, we define a simple cosine shaped tone generator $A \cos(\omega t)$

In [ ]:
tone = CosTone()
gen = CosGenerator(devices=[tone])

We can inspect the pre-defined parameters with

In [ ]:
params = tone.getParameters()
params

in this case amplitude $A$ and frequency $\omega$.

Next, we setup the qubit system we want to control. We set the qubit frequency $\omega_q$ to be 4.8 GHz and define the Hamiltonian as $$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x$$, where $\Omega(t)$ will be supplied by the generator.

In [ ]:
FREQ = 4.8e9 * 2 * np.pi

drive = GeneratorDrive(gen, isLongitudinal=False)
controlled_qubit = Qubit(
    frequency=Quantity(FREQ / 2, min_value=FREQ / 4, max_value=FREQ, unit="Hz", name="Qubit frequency"),
    drives=[drive]
)

model = ClosedModel(controlled_qubit)

Textbook values for implementing an $X$ rotation on this system at a time $T$ would be $\omega=\omega_q$ and $A=\pi/T$. We use some offset from these values as initial guess to demonstrate the optimization procedure.

In [ ]:
t_final = 10e-9
params[0].setValue(0.5 * np.pi / t_final)
params[1].setValue(1.01 * FREQ)

We select a propagation method, piecewise constant exponentation, and configure $\sigma_x$ as a target gate. Also we initialize the full basis at time 0 with $\mathcal{I}_2$

In [ ]:
prop = ScipyExpmGOAT(model, res=100e9)

xGate = np.array(
    [[0.0, 1], 
     [1, 0.0]]
    )  

prop.setInitialState(np.identity(2))
gateFid = UnitaryFidelity(
    propagation=prop,
    gate=xGate,
    times=np.array([0.0, t_final]),
)

In [ ]:
def make_plot():
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts/1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts/1e-9, np.abs(states)[:,:,0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax

make_plot()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
gateFid.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone.

In [ ]:
optmap = OptimisationMap()
optmap.add(tone, params)
opt = ScipyOptimiserGradient(gateFid, optimisables=optmap)

In [ ]:
opt.optimise()

In [ ]:
make_plot()

We can see from the plot and optimizer output that we have found good controls.

In [ ]:
gateFid.measure()

In [ ]:
params